In [1]:
import sys, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.models.embeddings import user_profiles_dense, score_dense
from src.models.hybrid import zscore, weighted

cfg = load_config("small")
spark = get_spark(cfg)
OUT = cfg["paths"]["processed"]
RES = ROOT / "results" / "metrics"

ALPHAS = (20, 100)
WS = (0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0)

imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))
news_emb = spark.read.parquet(str(OUT / "news_emb_minilm")).cache()

In [2]:
def senales_estaticas(periodos_ctr, periodo_eval):
    """CTR acumulado (fijo) con los periodos dados + coseno de embeddings, sobre el periodo evaluado."""
    hist = imp.filter(F.col("periodo").isin(*periodos_ctr))
    p0 = hist.agg(F.avg("label")).first()[0]
    conteos = hist.groupBy("news_id").agg(F.sum("label").alias("clics"), F.count("*").alias("expos"))

    cands = (imp.filter(F.col("periodo") == periodo_eval)
                .select("impression_id", "user_id", "news_id", "label")
                .join(conteos, "news_id", "left")
                .fillna(0, subset=["clics", "expos"]))
    for a in ALPHAS:
        cands = cands.withColumn(f"ctr_a{a}", (F.col("clics") + a * p0) / (F.col("expos") + a))

    users = (beh.filter(F.col("periodo") == periodo_eval)
                .select("user_id", "history").dropDuplicates(["user_id"]))
    s = score_dense(cands, news_emb, user_profiles_dense(users, news_emb, k=None))

    s = s.withColumn("cos_z", zscore("cos"))
    for a in ALPHAS:
        s = s.withColumn(f"ctr_a{a}_z", zscore(f"ctr_a{a}"))
    return s.drop("emb", "u_emb").cache(), p0

In [3]:
t0 = time.time()
val_s, p0_val = senales_estaticas(["train"], "val")
print(f"p0 = {p0_val:.4f} | candidatas de val: {val_s.count():,} ({time.time() - t0:.0f} s)")

val_s.agg(F.round(F.avg((F.col("expos") == 0).cast("int")), 3).alias("cand_sin_ctr"),
          F.round(F.sum(F.col("label") * (F.col("expos") == 0).cast("int")) / F.sum("label"), 3)
           .alias("clics_sin_ctr")).show()

p0 = 0.0410 | candidatas de val: 1,222,429 (213 s)
+------------+-------------+
|cand_sin_ctr|clics_sin_ctr|
+------------+-------------+
|       0.292|        0.369|
+------------+-------------+



In [4]:
def eval_score(df, expr, nombre):
    per_imp, res = evaluate(df.withColumn("score", expr))
    per_imp.unpersist()
    return {"modelo": nombre, **res}

filas = []
for a in ALPHAS:
    for w in WS:
        nombre = f"estatico_a{a}_w{w:.1f}"
        filas.append(eval_score(val_s, weighted("cos_z", f"ctr_a{a}_z", w), nombre))
        print(f"{nombre:20s} AUC = {filas[-1]['auc']:.4f} | nDCG@5 = {filas[-1]['ndcg5']:.4f}")

tabla_val = (pd.DataFrame(filas)[["modelo", "auc", "mrr", "ndcg5", "ndcg10"]]
             .sort_values("auc", ascending=False).reset_index(drop=True))
mejor = tabla_val.iloc[0]
print(f"\nMejor en val: {mejor['modelo']}  AUC = {mejor['auc']:.4f}")
tabla_val.round(4)

estatico_a20_w0.0    AUC = 0.5739 | nDCG@5 = 0.2411
estatico_a20_w0.2    AUC = 0.6352 | nDCG@5 = 0.2757
estatico_a20_w0.4    AUC = 0.6488 | nDCG@5 = 0.2926
estatico_a20_w0.5    AUC = 0.6526 | nDCG@5 = 0.3033
estatico_a20_w0.6    AUC = 0.6532 | nDCG@5 = 0.3160
estatico_a20_w0.8    AUC = 0.6481 | nDCG@5 = 0.3232
estatico_a20_w1.0    AUC = 0.6352 | nDCG@5 = 0.3139
estatico_a100_w0.0   AUC = 0.5734 | nDCG@5 = 0.2417
estatico_a100_w0.2   AUC = 0.6366 | nDCG@5 = 0.2788
estatico_a100_w0.4   AUC = 0.6472 | nDCG@5 = 0.2935
estatico_a100_w0.5   AUC = 0.6500 | nDCG@5 = 0.3029
estatico_a100_w0.6   AUC = 0.6508 | nDCG@5 = 0.3140
estatico_a100_w0.8   AUC = 0.6470 | nDCG@5 = 0.3223
estatico_a100_w1.0   AUC = 0.6352 | nDCG@5 = 0.3139

Mejor en val: estatico_a20_w0.6  AUC = 0.6532


,modelo,auc,mrr,ndcg5,ndcg10
0,estatico_a20_w0.6,0.6532,0.2945,0.3160,0.3788
1,estatico_a20_w0.5,0.6526,0.2826,0.3033,0.3685
2,estatico_a100_w0.6,0.6508,0.2923,0.3140,0.3769
3,estatico_a100_w0.5,0.6500,0.2814,0.3029,0.3672
4,estatico_a20_w0.4,0.6488,0.2733,0.2926,0.3590
5,estatico_a20_w0.8,0.6481,0.2999,0.3232,0.3834
6,estatico_a100_w0.4,0.6472,0.2734,0.2935,0.3592
7,estatico_a100_w0.8,0.6470,0.2993,0.3223,0.3829
8,estatico_a100_w0.2,0.6366,0.2638,0.2788,0.3457
9,estatico_a20_w0.2,0.6352,0.2621,0.2757,0.3426


In [5]:
a_best = int(mejor["modelo"].split("_a")[1].split("_")[0])
w_best = float(mejor["modelo"].split("_w")[1])

test_s, p0_test = senales_estaticas(["train", "val"], "test")
estim = [
    eval_score(test_s, weighted("cos_z", f"ctr_a{a_best}_z", w_best), f"estatico_a{a_best}_w{w_best:.1f}"),
    eval_score(test_s, F.col(f"ctr_a{a_best}"), "solo_ctr_estatico"),
]
print(f"Configuración para CodaLab: alpha = {a_best}, w = {w_best}")
pd.DataFrame(estim)[["modelo", "auc", "mrr", "ndcg5", "ndcg10"]].round(4)

Configuración para CodaLab: alpha = 20, w = 0.6


,modelo,auc,mrr,ndcg5,ndcg10
0,estatico_a20_w0.6,0.6555,0.3066,0.3385,0.3998
1,solo_ctr_estatico,0.5854,0.2672,0.2828,0.3443


In [6]:
tabla_val.assign(periodo="val").to_csv(RES / "09_sin_clics_val.csv", index=False)
pd.DataFrame(estim).assign(periodo="test").to_csv(RES / "09_sin_clics_test.csv", index=False)
spark.stop()